In [8]:
!pip install -q pytorch-tabnet==4.1.0
!pip install -q optuna==3.5.0

from pytorch_tabnet.tab_model import TabNetClassifier, TabNetRegressor
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)
print("✅ OK")

✅ OK


In [9]:
import numpy as np
import pandas as pd
import pickle, os, time, warnings
warnings.filterwarnings('ignore')

import torch
from sklearn.datasets import fetch_california_housing, fetch_covtype
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import (mean_squared_error, mean_absolute_error, r2_score,
                             accuracy_score, roc_auc_score, f1_score)

os.makedirs('/kaggle/working/results', exist_ok=True)
SEEDS = [0, 42, 123]
N_TRIALS = 20
print(f"✅ GPU available: {torch.cuda.is_available()}")

✅ GPU available: True


In [10]:
# ── California Housing ────────────────────────────────
raw = fetch_california_housing(as_frame=True)
df_cal = raw.frame.copy()

# ── Adult Income ──────────────────────────────────────
col_names = ['age','workclass','fnlwgt','education','education_num',
             'marital_status','occupation','relationship','race','sex',
             'capital_gain','capital_loss','hours_per_week','native_country','income']
df_a1 = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.data",
                    header=None, names=col_names, skipinitialspace=True, na_values='?')
df_a2 = pd.read_csv("https://archive.ics.uci.edu/ml/machine-learning-databases/adult/adult.test",
                    header=None, names=col_names, skipinitialspace=True, na_values='?', skiprows=1)
df_adult = pd.concat([df_a1, df_a2], ignore_index=True)
df_adult['income'] = df_adult['income'].str.replace('.','',regex=False).str.strip()
df_adult['income'] = (df_adult['income'] == '>50K').astype(int)

# ── Covertype ─────────────────────────────────────────
raw_cov = fetch_covtype(as_frame=True)
df_cov = raw_cov.frame.copy()
df_cov = df_cov.sample(n=20_000, random_state=42).reset_index(drop=True)
df_cov['Cover_Type'] = df_cov['Cover_Type'] - 1

print("✅ Datasets loaded")
print(f"  California: {df_cal.shape}, Adult: {df_adult.shape}, Covertype: {df_cov.shape}")

✅ Datasets loaded
  California: (20640, 9), Adult: (48842, 15), Covertype: (20000, 55)


In [11]:
def preprocess_dataset(df, target_col, task, seed=42):
    df = df.copy()
    X = df.drop(columns=[target_col])
    y = df[target_col].values

    cat_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()
    num_cols = X.select_dtypes(include=['number']).columns.tolist()

    for col in cat_cols:
        X[col] = X[col].fillna('__missing__')
        X[col] = LabelEncoder().fit_transform(X[col].astype(str))
    for col in num_cols:
        X[col] = X[col].fillna(X[col].median())

    feature_cols = num_cols + cat_cols
    X = X[feature_cols].values.astype(np.float32)

    X_tr, X_temp, y_tr, y_temp = train_test_split(
        X, y, test_size=0.40, random_state=seed,
        stratify=y if task=='classification' else None)
    X_val, X_te, y_val, y_te = train_test_split(
        X_temp, y_temp, test_size=0.50, random_state=seed,
        stratify=y_temp if task=='classification' else None)

    n_num = len(num_cols)
    scaler = StandardScaler()
    X_tr[:, :n_num]  = scaler.fit_transform(X_tr[:, :n_num])
    X_val[:, :n_num] = scaler.transform(X_val[:, :n_num])
    X_te[:, :n_num]  = scaler.transform(X_te[:, :n_num])

    cat_indices = list(range(n_num, n_num + len(cat_cols)))
    cat_dims = [int(X_tr[:, i].max()) + 2 for i in cat_indices]  # TabNet需要

    return X_tr, X_val, X_te, \
           y_tr.astype(np.float32) if task=='regression' else y_tr.astype(np.int64), \
           y_val.astype(np.float32) if task=='regression' else y_val.astype(np.int64), \
           y_te.astype(np.float32)  if task=='regression' else y_te.astype(np.int64), \
           feature_cols, cat_indices, cat_dims

def evaluate(y_true, y_pred, y_prob, task):
    metrics = {}
    if task == 'regression':
        metrics['RMSE'] = np.sqrt(mean_squared_error(y_true, y_pred))
        metrics['MAE']  = mean_absolute_error(y_true, y_pred)
        metrics['R2']   = r2_score(y_true, y_pred)
    elif task == 'classification':
        metrics['Accuracy'] = accuracy_score(y_true, y_pred)
        metrics['AUC']      = roc_auc_score(y_true, y_prob[:, 1])
        metrics['F1']       = f1_score(y_true, y_pred, average='binary')
    elif task == 'multiclass':
        metrics['Accuracy'] = accuracy_score(y_true, y_pred)
        metrics['AUC']      = roc_auc_score(y_true, y_prob, multi_class='ovr', average='macro')
        metrics['F1']       = f1_score(y_true, y_pred, average='macro')
    return metrics

print("✅ Functions defined")

✅ Functions defined


In [12]:
print("📦 Preprocessing datasets...")

cal_splits   = preprocess_dataset(df_cal,   'MedHouseVal', 'regression')
adult_splits = preprocess_dataset(df_adult, 'income',      'classification')
cov_splits   = preprocess_dataset(df_cov,   'Cover_Type',  'multiclass')

datasets = {
    'california': {'splits': cal_splits,   'task': 'regression',     'n_classes': 1},
    'adult':      {'splits': adult_splits, 'task': 'classification',  'n_classes': 2},
    'covertype':  {'splits': cov_splits,   'task': 'multiclass',      'n_classes': 7},
}
print("✅ Done")

📦 Preprocessing datasets...
✅ Done


In [13]:
def run_tabnet(dataset_name, datasets, seeds, n_trials=20):
    d = datasets[dataset_name]
    task = d['task']
    X_tr, X_val, X_te, y_tr, y_val, y_te, feat_names, cat_indices, cat_dims = d['splits']
    n_classes = d['n_classes']

    all_metrics = []
    best_params_list = []

    for seed in seeds:
        print(f"\n  🌱 seed={seed}")
        torch.manual_seed(seed)
        np.random.seed(seed)

        def objective(trial):
            n_d       = trial.suggest_int('n_d', 8, 64)
            n_steps   = trial.suggest_int('n_steps', 3, 10)
            gamma     = trial.suggest_float('gamma', 1.0, 2.0)
            momentum  = trial.suggest_float('momentum', 0.01, 0.4)
            lr        = trial.suggest_float('lr', 1e-4, 1e-2, log=True)
            batch_size= trial.suggest_categorical('batch_size', [256, 512, 1024])

            common = dict(
                n_d=n_d, n_a=n_d, n_steps=n_steps, gamma=gamma,
                momentum=momentum, cat_idxs=cat_indices, cat_dims=cat_dims,
                verbose=0, seed=seed, device_name='cuda',
                optimizer_fn=torch.optim.Adam,
                optimizer_params={'lr': lr},
            )
            fit_common = dict(
                max_epochs=100, patience=15,
                batch_size=batch_size,
                virtual_batch_size=batch_size // 4,
            )

            if task == 'regression':
                model = TabNetRegressor(**common)
                model.fit(X_tr, y_tr.reshape(-1, 1),
                          eval_set=[(X_val, y_val.reshape(-1, 1))],
                          eval_metric=['rmse'], **fit_common)
                preds = model.predict(X_val).flatten()
                return np.sqrt(mean_squared_error(y_val, preds))
            else:
                model = TabNetClassifier(**common)
                model.fit(X_tr, y_tr,
                          eval_set=[(X_val, y_val)],
                          eval_metric=['accuracy'], **fit_common)
                preds = model.predict(X_val)
                return -accuracy_score(y_val, preds)

        study = optuna.create_study(direction='minimize',
                                    sampler=optuna.samplers.TPESampler(seed=seed))
        study.optimize(objective, n_trials=n_trials, show_progress_bar=True)

        best = study.best_params
        best_params_list.append(best)
        print(f"    Best params: {best}")

        # 用最优参数在 train+val 上重训，test 上评估
        common_final = dict(
            n_d=best['n_d'], n_a=best['n_d'],
            n_steps=best['n_steps'], gamma=best['gamma'],
            momentum=best['momentum'],
            cat_idxs=cat_indices, cat_dims=cat_dims,
            verbose=0, seed=seed, device_name='cuda',
            optimizer_fn=torch.optim.Adam,
            optimizer_params={'lr': best['lr']},
        )
        fit_final = dict(
            max_epochs=150, patience=20,
            batch_size=best['batch_size'],
            virtual_batch_size=best['batch_size'] // 4,
        )

        if task == 'regression':
            final_model = TabNetRegressor(**common_final)
            final_model.fit(
                np.concatenate([X_tr, X_val]),
                np.concatenate([y_tr, y_val]).reshape(-1, 1),
                **fit_final)
            y_pred = final_model.predict(X_te).flatten()
            y_prob = None
        else:
            final_model = TabNetClassifier(**common_final)
            final_model.fit(
                np.concatenate([X_tr, X_val]),
                np.concatenate([y_tr, y_val]),
                **fit_final)
            y_prob = final_model.predict_proba(X_te)
            y_pred = np.argmax(y_prob, axis=1)

        m = evaluate(y_te, y_pred, y_prob, task)
        m['seed'] = seed
        all_metrics.append(m)
        print(f"    Test metrics: {m}")

        with open(f'/kaggle/working/results/tabnet_{dataset_name}_seed{seed}.pkl', 'wb') as f:
            pickle.dump({'metrics': m, 'best_params': best}, f)

    df_m = pd.DataFrame(all_metrics).drop(columns='seed')
    summary = {col: f"{df_m[col].mean():.4f} ± {df_m[col].std():.4f}"
               for col in df_m.columns}
    return summary, best_params_list

print("✅ run_tabnet() defined")

✅ run_tabnet() defined


In [14]:
print("=" * 55)
print("🔷 TabNet — CALIFORNIA HOUSING")
print("=" * 55)
t0 = time.time()

cal_summary, cal_params = run_tabnet('california', datasets, SEEDS, N_TRIALS)

print(f"\n✅ Done in {(time.time()-t0)/60:.1f} min")
print(f"Summary: {cal_summary}")

🔷 TabNet — CALIFORNIA HOUSING

  🌱 seed=0


  0%|          | 0/20 [00:00<?, ?it/s]


Early stopping occurred at epoch 51 with best_epoch = 36 and best_val_0_rmse = 0.83109
Stop training because you reached max_epochs = 100 with best_epoch = 99 and best_val_0_rmse = 0.69002

Early stopping occurred at epoch 81 with best_epoch = 66 and best_val_0_rmse = 0.64792
Stop training because you reached max_epochs = 100 with best_epoch = 90 and best_val_0_rmse = 0.74838
Stop training because you reached max_epochs = 100 with best_epoch = 95 and best_val_0_rmse = 0.64559
Stop training because you reached max_epochs = 100 with best_epoch = 86 and best_val_0_rmse = 0.67436

Early stopping occurred at epoch 55 with best_epoch = 40 and best_val_0_rmse = 0.67265
Stop training because you reached max_epochs = 100 with best_epoch = 97 and best_val_0_rmse = 0.76718

Early stopping occurred at epoch 72 with best_epoch = 57 and best_val_0_rmse = 0.66283

Early stopping occurred at epoch 66 with best_epoch = 51 and best_val_0_rmse = 0.80097

Early stopping occurred at epoch 59 with best_epo

  0%|          | 0/20 [00:00<?, ?it/s]


Early stopping occurred at epoch 97 with best_epoch = 82 and best_val_0_rmse = 1.01267

Early stopping occurred at epoch 55 with best_epoch = 40 and best_val_0_rmse = 0.63935
Stop training because you reached max_epochs = 100 with best_epoch = 92 and best_val_0_rmse = 0.70889

Early stopping occurred at epoch 92 with best_epoch = 77 and best_val_0_rmse = 0.70305

Early stopping occurred at epoch 91 with best_epoch = 76 and best_val_0_rmse = 0.98911

Early stopping occurred at epoch 94 with best_epoch = 79 and best_val_0_rmse = 0.7693

Early stopping occurred at epoch 36 with best_epoch = 21 and best_val_0_rmse = 0.65837
Stop training because you reached max_epochs = 100 with best_epoch = 99 and best_val_0_rmse = 0.69393

Early stopping occurred at epoch 95 with best_epoch = 80 and best_val_0_rmse = 0.81639

Early stopping occurred at epoch 56 with best_epoch = 41 and best_val_0_rmse = 0.76586

Early stopping occurred at epoch 45 with best_epoch = 30 and best_val_0_rmse = 0.69226

Earl

  0%|          | 0/20 [00:00<?, ?it/s]


Early stopping occurred at epoch 64 with best_epoch = 49 and best_val_0_rmse = 0.64097
Stop training because you reached max_epochs = 100 with best_epoch = 87 and best_val_0_rmse = 0.73019

Early stopping occurred at epoch 65 with best_epoch = 50 and best_val_0_rmse = 0.67617

Early stopping occurred at epoch 93 with best_epoch = 78 and best_val_0_rmse = 0.7201

Early stopping occurred at epoch 84 with best_epoch = 69 and best_val_0_rmse = 0.72942

Early stopping occurred at epoch 42 with best_epoch = 27 and best_val_0_rmse = 0.64191

Early stopping occurred at epoch 92 with best_epoch = 77 and best_val_0_rmse = 0.65764

Early stopping occurred at epoch 93 with best_epoch = 78 and best_val_0_rmse = 0.72186
Stop training because you reached max_epochs = 100 with best_epoch = 96 and best_val_0_rmse = 0.68641
Stop training because you reached max_epochs = 100 with best_epoch = 90 and best_val_0_rmse = 0.67353

Early stopping occurred at epoch 23 with best_epoch = 8 and best_val_0_rmse = 

In [7]:
print("=" * 55)
print("🔷 TabNet — ADULT INCOME")
print("=" * 55)
t0 = time.time()

adult_summary, adult_params = run_tabnet('adult', datasets, SEEDS, N_TRIALS)

print(f"\n✅ Done in {(time.time()-t0)/60:.1f} min")
print(f"Summary: {adult_summary}")

🔷 TabNet — ADULT INCOME

  🌱 seed=0


  0%|          | 0/20 [00:00<?, ?it/s]

Stop training because you reached max_epochs = 100 with best_epoch = 85 and best_val_0_accuracy = 0.84459

Early stopping occurred at epoch 86 with best_epoch = 71 and best_val_0_accuracy = 0.84777

Early stopping occurred at epoch 84 with best_epoch = 69 and best_val_0_accuracy = 0.85708

Early stopping occurred at epoch 49 with best_epoch = 34 and best_val_0_accuracy = 0.84337

Early stopping occurred at epoch 37 with best_epoch = 22 and best_val_0_accuracy = 0.85412

Early stopping occurred at epoch 34 with best_epoch = 19 and best_val_0_accuracy = 0.84029

Early stopping occurred at epoch 26 with best_epoch = 11 and best_val_0_accuracy = 0.84623
Stop training because you reached max_epochs = 100 with best_epoch = 95 and best_val_0_accuracy = 0.84818
Stop training because you reached max_epochs = 100 with best_epoch = 96 and best_val_0_accuracy = 0.85637

Early stopping occurred at epoch 96 with best_epoch = 81 and best_val_0_accuracy = 0.83835

Early stopping occurred at epoch 84 w

  0%|          | 0/20 [00:00<?, ?it/s]

Stop training because you reached max_epochs = 100 with best_epoch = 98 and best_val_0_accuracy = 0.82514

Early stopping occurred at epoch 39 with best_epoch = 24 and best_val_0_accuracy = 0.85268

Early stopping occurred at epoch 57 with best_epoch = 42 and best_val_0_accuracy = 0.84398

Early stopping occurred at epoch 65 with best_epoch = 50 and best_val_0_accuracy = 0.84419

Early stopping occurred at epoch 62 with best_epoch = 47 and best_val_0_accuracy = 0.80938
Stop training because you reached max_epochs = 100 with best_epoch = 88 and best_val_0_accuracy = 0.84808

Early stopping occurred at epoch 62 with best_epoch = 47 and best_val_0_accuracy = 0.84828

Early stopping occurred at epoch 99 with best_epoch = 84 and best_val_0_accuracy = 0.85125

Early stopping occurred at epoch 81 with best_epoch = 66 and best_val_0_accuracy = 0.8318

Early stopping occurred at epoch 63 with best_epoch = 48 and best_val_0_accuracy = 0.84214

Early stopping occurred at epoch 62 with best_epoch 

  0%|          | 0/20 [00:00<?, ?it/s]


Early stopping occurred at epoch 40 with best_epoch = 25 and best_val_0_accuracy = 0.85432

Early stopping occurred at epoch 81 with best_epoch = 66 and best_val_0_accuracy = 0.84787

Early stopping occurred at epoch 67 with best_epoch = 52 and best_val_0_accuracy = 0.85452

Early stopping occurred at epoch 45 with best_epoch = 30 and best_val_0_accuracy = 0.84541

Early stopping occurred at epoch 69 with best_epoch = 54 and best_val_0_accuracy = 0.8448

Early stopping occurred at epoch 43 with best_epoch = 28 and best_val_0_accuracy = 0.85913

Early stopping occurred at epoch 89 with best_epoch = 74 and best_val_0_accuracy = 0.85627

Early stopping occurred at epoch 46 with best_epoch = 31 and best_val_0_accuracy = 0.8364

Early stopping occurred at epoch 92 with best_epoch = 77 and best_val_0_accuracy = 0.84818

Early stopping occurred at epoch 50 with best_epoch = 35 and best_val_0_accuracy = 0.85401

Early stopping occurred at epoch 28 with best_epoch = 13 and best_val_0_accuracy 

In [15]:
print("=" * 55)
print("🔷 TabNet — COVERTYPE")
print("=" * 55)
t0 = time.time()

cov_summary, cov_params = run_tabnet('covertype', datasets, SEEDS, N_TRIALS)

print(f"\n✅ Done in {(time.time()-t0)/60:.1f} min")
print(f"Summary: {cov_summary}")

🔷 TabNet — COVERTYPE

  🌱 seed=0


  0%|          | 0/20 [00:00<?, ?it/s]


Early stopping occurred at epoch 73 with best_epoch = 58 and best_val_0_accuracy = 0.66025

Early stopping occurred at epoch 61 with best_epoch = 46 and best_val_0_accuracy = 0.6955

Early stopping occurred at epoch 80 with best_epoch = 65 and best_val_0_accuracy = 0.73025
Stop training because you reached max_epochs = 100 with best_epoch = 90 and best_val_0_accuracy = 0.69125
Stop training because you reached max_epochs = 100 with best_epoch = 95 and best_val_0_accuracy = 0.76475

Early stopping occurred at epoch 57 with best_epoch = 42 and best_val_0_accuracy = 0.706

Early stopping occurred at epoch 61 with best_epoch = 46 and best_val_0_accuracy = 0.7255

Early stopping occurred at epoch 98 with best_epoch = 83 and best_val_0_accuracy = 0.65825
Stop training because you reached max_epochs = 100 with best_epoch = 99 and best_val_0_accuracy = 0.777

Early stopping occurred at epoch 94 with best_epoch = 79 and best_val_0_accuracy = 0.66575
Stop training because you reached max_epochs

  0%|          | 0/20 [00:00<?, ?it/s]


Early stopping occurred at epoch 62 with best_epoch = 47 and best_val_0_accuracy = 0.458
Stop training because you reached max_epochs = 100 with best_epoch = 98 and best_val_0_accuracy = 0.7465
Stop training because you reached max_epochs = 100 with best_epoch = 87 and best_val_0_accuracy = 0.71025

Early stopping occurred at epoch 66 with best_epoch = 51 and best_val_0_accuracy = 0.69625

Early stopping occurred at epoch 75 with best_epoch = 60 and best_val_0_accuracy = 0.462
Stop training because you reached max_epochs = 100 with best_epoch = 88 and best_val_0_accuracy = 0.67875

Early stopping occurred at epoch 92 with best_epoch = 77 and best_val_0_accuracy = 0.7785
Stop training because you reached max_epochs = 100 with best_epoch = 89 and best_val_0_accuracy = 0.71275

Early stopping occurred at epoch 93 with best_epoch = 78 and best_val_0_accuracy = 0.6305

Early stopping occurred at epoch 68 with best_epoch = 53 and best_val_0_accuracy = 0.626

Early stopping occurred at epoch

  0%|          | 0/20 [00:00<?, ?it/s]

Stop training because you reached max_epochs = 100 with best_epoch = 94 and best_val_0_accuracy = 0.77625

Early stopping occurred at epoch 35 with best_epoch = 20 and best_val_0_accuracy = 0.55425

Early stopping occurred at epoch 83 with best_epoch = 68 and best_val_0_accuracy = 0.7385

Early stopping occurred at epoch 56 with best_epoch = 41 and best_val_0_accuracy = 0.67475

Early stopping occurred at epoch 84 with best_epoch = 69 and best_val_0_accuracy = 0.68975

Early stopping occurred at epoch 92 with best_epoch = 77 and best_val_0_accuracy = 0.79125

Early stopping occurred at epoch 51 with best_epoch = 36 and best_val_0_accuracy = 0.6905

Early stopping occurred at epoch 69 with best_epoch = 54 and best_val_0_accuracy = 0.67775
Stop training because you reached max_epochs = 100 with best_epoch = 99 and best_val_0_accuracy = 0.699
Stop training because you reached max_epochs = 100 with best_epoch = 94 and best_val_0_accuracy = 0.72675

Early stopping occurred at epoch 37 with 

In [16]:
tabnet_results = {
    'california': cal_summary,
    'adult':      adult_summary,
    'covertype':  cov_summary,
}

with open('/kaggle/working/results/tabnet_results.pkl', 'wb') as f:
    pickle.dump(tabnet_results, f)

print("\n" + "="*55)
print("📊 TABNET RESULTS SUMMARY")
print("="*55)
for ds, summary in tabnet_results.items():
    print(f"\n── {ds.upper()} ──")
    for metric, val in summary.items():
        print(f"  {metric}: {val}")

print("\n✅ All TabNet results saved!")


📊 TABNET RESULTS SUMMARY

── CALIFORNIA ──
  RMSE: 0.6645 ± 0.0081
  MAE: 0.4379 ± 0.0057
  R2: 0.6779 ± 0.0078

── ADULT ──
  Accuracy: 0.8536 ± 0.0047
  AUC: 0.9042 ± 0.0077
  F1: 0.6553 ± 0.0050

── COVERTYPE ──
  Accuracy: 0.8191 ± 0.0110
  AUC: 0.9623 ± 0.0049
  F1: 0.7014 ± 0.0358

✅ All TabNet results saved!


In [17]:
rows = []
for ds, summary in tabnet_results.items():
    for metric, val in summary.items():
        rows.append({'dataset': ds, 'model': 'TabNet', 'metric': metric, 'value': val})
pd.DataFrame(rows).to_csv('/kaggle/working/results/tabnet_results.csv', index=False)
print(pd.DataFrame(rows).to_string())

      dataset   model    metric            value
0  california  TabNet      RMSE  0.6645 ± 0.0081
1  california  TabNet       MAE  0.4379 ± 0.0057
2  california  TabNet        R2  0.6779 ± 0.0078
3       adult  TabNet  Accuracy  0.8536 ± 0.0047
4       adult  TabNet       AUC  0.9042 ± 0.0077
5       adult  TabNet        F1  0.6553 ± 0.0050
6   covertype  TabNet  Accuracy  0.8191 ± 0.0110
7   covertype  TabNet       AUC  0.9623 ± 0.0049
8   covertype  TabNet        F1  0.7014 ± 0.0358
